# Eval
this notebook is for calling supporting scripts to derive and present evaluations in an accessible fashion. 

# 1 Imports & Setup

In [1]:
import sys, pathlib

PROJECT_ROOT = pathlib.Path('.').resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torchaudio
from transformers import AutoProcessor, AutoModelForCTC, pipeline


/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from scripts.eval.fetch_annotations import fetch_annotations
from scripts.eval.eval import evaluate_mispronunciation_detection
from scripts.data_handling.collapse_phonemes import collapse_phones
from scripts.asr_system.ensemble.run_inference import run_ensemble_inference
from scripts.asr_system.ensemble.ensemble import (
    build_ru_ipa_dict,
    build_pal_set,
    frame_gibbs_confidence,
)

In [4]:
DEVICE    =  "cpu" # "cuda" if torch.cuda.is_available() else "cpu" decomment if running on kaggle.
AUDIO_DIR = PROJECT_ROOT / "data" / "fleurs" / "audio" / "test" # point to kaggle dataset if using on there.
ANN_CACHE = PROJECT_ROOT / "data" / "eval" / "annotations_cache.json" # point to kaggle dataset.
IMAGES_DIR = PROJECT_ROOT / "paper" / "0000-CadyThesis-Draft" / "images"

GA_MODEL_ID           = "duck-hug-567/xls-r-300m-gaphon-full-collapsed"
EN_MODEL_ID           = "duck-hug-567/xls-r-300m-engphon-collapsed"
RU_MODEL_ID           = "snu-nia-12/wav2vec-large-xlsr-53_nia12_phone-nsu-ai-_russian"  # set to model ID string to enable Russian specialist
L2_MODEL_ID           = "duck-hug-567/xls-r-300m-paired"  # synthetic-augmented model (paired training)
L2_BASELINE_MODEL_ID  = None  # synthetic-augmented model (unpaired baseline)
COMBINED_MODEL_ID  = None  # single model trained on Irish+English data combined

print(f"Device:     {DEVICE}")
print(f"Images dir: {IMAGES_DIR}")

Device:     cpu
Images dir: /home/peter/Desktop/thesis/ThesisProject/paper/0000-CadyThesis-Draft/images


In [6]:
ga_processor = AutoProcessor.from_pretrained(GA_MODEL_ID)
ga_model     = AutoModelForCTC.from_pretrained(GA_MODEL_ID).to(DEVICE).eval()

en_processor = AutoProcessor.from_pretrained(EN_MODEL_ID)
en_model     = AutoModelForCTC.from_pretrained(EN_MODEL_ID).to(DEVICE).eval()

ru_processor = ru_model = None
if RU_MODEL_ID:
    ru_processor = AutoProcessor.from_pretrained(RU_MODEL_ID)
    ru_model     = AutoModelForCTC.from_pretrained(RU_MODEL_ID).to(DEVICE).eval()

# Irish-only pipeline — reuses the already-loaded ga_model, no extra download
ga_pipe = pipeline(
    "automatic-speech-recognition",
    model=ga_model,
    tokenizer=ga_processor.tokenizer,
    feature_extractor=ga_processor.feature_extractor,
    device=DEVICE,
)

l2_pipe = (pipeline("automatic-speech-recognition", model=L2_MODEL_ID, device=DEVICE)
           if L2_MODEL_ID else None)

l2_baseline_pipe = (pipeline("automatic-speech-recognition", model=L2_BASELINE_MODEL_ID, device=DEVICE)
                    if L2_BASELINE_MODEL_ID else None)

combined_pipe = (pipeline("automatic-speech-recognition", model=COMBINED_MODEL_ID, device=DEVICE)
                 if COMBINED_MODEL_ID else None)

print("Models loaded.")

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 761.55it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


Models loaded.


# 2 Load Data
Annotations (FLEURS Gold standard), Ulster transcriptions (canonical), audio (FLEURS eval audio)

In [7]:
records = fetch_annotations(cache_path=ANN_CACHE, refresh=False)

print(f"Loaded {len(records)} annotated utterances")
print(f"Total canonical phones : {sum(len(r['canonical']) for r in records)}")
print(f"Total gold phones      : {sum(len(r['gold']) for r in records)}")

Loaded 5 annotated utterances
Total canonical phones : 622
Total gold phones      : 667


## 2.1 Normalise

Apply `collapse_phones` to canonical and gold so all three sequences (canonical, gold, asr)
share the same phone space — the ga model's collapsed vocab.
The asr output from inference is already in this space by construction.

In [8]:
for r in records:
    r['canonical'] = collapse_phones(r['canonical'])
    r['gold']      = collapse_phones(r['gold'])

worth noting that this normalization is performed in place, so if I need access to the originals, I need ot reload the fetching.

## 2.2 Sanity check

In [9]:
for r in records:
    print(f"\n{r['audio_id']}")
    print(f"  canonical ({len(r['canonical'])}): {' '.join(r['canonical'][:10])} ...")
    print(f"  gold      ({len(r['gold'])}): {' '.join(r['gold'][:10])} ...")


10045899353945045473
  canonical (104): i s eː pʲ ɾʲ iː v x a h ...
  gold      (121): i s eː ɑ w ɑ ʃ ə a n ...

10090378450980723387
  canonical (54): i s ia d n ə lʲ oː nʲ n ...
  gold      (51): i s i d lʲ oi nʲ n a x ...

10108759754831048301
  canonical (85): dʲ eː ɾ eː ɾʲ m a ɾ ə h ...
  gold      (86): dʲ eː ɹ eː ɹ m a ɾ ə h ...

10113135482749057387
  canonical (219): h ua ɾʲ i ʃ c iː t ai dʲ ...
  gold      (242): h ua ɾʲ s iː t ai dʲ oː ɾʲ ...

10118867771707376132
  canonical (160): ɡ ə ɟ i nʲ ə ɾ a l t ...
  gold      (167): ɣ ɑ ɣ e nʲ ə ɾ a l t ...


# 3 Inference

## 3.1 Ensemble

In [ ]:
ensemble_2way_records = run_ensemble_inference(
    records,
    ga_processor, ga_model,
    en_processor, en_model,
    ru_processor=None, ru_model=None,
    audio_dir=AUDIO_DIR,
    device=DEVICE,
)
print(f"2-way done. {len(ensemble_2way_records)} utterances.")

ensemble_2way_pooled_records = run_ensemble_inference(
    records,
    ga_processor, ga_model,
    en_processor, en_model,
    ru_processor=None, ru_model=None,
    pool_ga=True,
    audio_dir=AUDIO_DIR,
    device=DEVICE,
)
print(f"2-way (pooled) done. {len(ensemble_2way_pooled_records)} utterances.")

ensemble_3way_records = None
ensemble_3way_pooled_records = None
if ru_processor is not None:
    ensemble_3way_records = run_ensemble_inference(
        records,
        ga_processor, ga_model,
        en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        audio_dir=AUDIO_DIR,
        device=DEVICE,
    )
    print(f"3-way done. {len(ensemble_3way_records)} utterances.")

    ensemble_3way_pooled_records = run_ensemble_inference(
        records,
        ga_processor, ga_model,
        en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        pool_ga=True,
        audio_dir=AUDIO_DIR,
        device=DEVICE,
    )
    print(f"3-way (pooled) done. {len(ensemble_3way_pooled_records)} utterances.")
else:
    print("RU_MODEL_ID not set — 3-way ensembles skipped.")

## 3.2 Synthetic-augmentation

### 3.2.1 Synthetic-augmented model (adversarial sampling)

In [ ]:
l2_records = None

if l2_pipe is not None:
    l2_records = []
    for r in records:
        audio_path = AUDIO_DIR / f"{r['audio_id']}.wav"
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        audio_input = {"array": waveform.squeeze(0).numpy(), "sampling_rate": 16000}

        raw = l2_pipe(audio_input)["text"]
        asr = collapse_phones(raw.split())

        result = dict(r)
        result["asr"] = asr
        l2_records.append(result)

    print(f"Done. {len(l2_records)} utterances inferred.")
else:
    print("L2_MODEL_ID not set — skipping. Set it in the config cell when the model is ready.")

### 3.2.2 Synthetic-augmented model (unpaired baseline)

Same architecture as 3.2.1 but trained on synthetic and canonical data as independent instances,
without adversarial canonical/L2 pairing. Serves as an ablation to isolate the contribution
of the pairing strategy.

In [ ]:
l2_baseline_records = None

if l2_baseline_pipe is not None:
    l2_baseline_records = []
    for r in records:
        audio_path = AUDIO_DIR / f"{r['audio_id']}.wav"
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        audio_input = {"array": waveform.squeeze(0).numpy(), "sampling_rate": 16000}

        raw = l2_baseline_pipe(audio_input)["text"]
        asr = collapse_phones(raw.split())

        result = dict(r)
        result["asr"] = asr
        l2_baseline_records.append(result)

    print(f"Done. {len(l2_baseline_records)} utterances inferred.")
else:
    print("L2_BASELINE_MODEL_ID not set — skipping.")

## 3.3 Baselines

### 3.3.1 Irish-only (no augmentation)

The un-augmented baseline: plain CTC greedy decode from the Irish model alone,
no ensemble, no synthetic data. This is the starting point the L2-gen approach
builds on — everything else should beat this or the experiment hasn't worked.

In [ ]:
irish_only_records = []
for r in records:
    audio_path = AUDIO_DIR / f"{r['audio_id']}.wav"
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    audio_input = {"array": waveform.squeeze(0).numpy(), "sampling_rate": 16000}

    raw = ga_pipe(audio_input)["text"]
    asr = collapse_phones(raw.split())

    result = dict(r)
    result["asr"] = asr
    irish_only_records.append(result)

print(f"Done. {len(irish_only_records)} utterances inferred.")

### 3.3.2 Irish+English combined (single model)

Single model trained on the union of Irish and English phoneme data, without
ensemble architecture. Tests whether diverse training data alone accounts for
any improvement, or whether the per-span confidence gating of the ensemble is
what matters. Requires a separately trained model — set `L2_COMBINED_MODEL_ID`
when ready.

In [ ]:
l2_combined_records = None

if combined_pipe is not None:
    l2_combined_records = []
    for r in records:
        audio_path = AUDIO_DIR / f"{r['audio_id']}.wav"
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        audio_input = {"array": waveform.squeeze(0).numpy(), "sampling_rate": 16000}

        raw = combined_pipe(audio_input)["text"]
        asr = collapse_phones(raw.split())

        result = dict(r)
        result["asr"] = asr
        l2_combined_records.append(result)

    print(f"Done. {len(l2_combined_records)} utterances inferred.")
else:
    print("COMBINED_MODEL_ID not set — skipping.")

# 4 Per item Eval

In [ ]:
# Central registry — add or comment out systems here as models become available.
# Any system whose records are None is automatically skipped below.
SYSTEMS = {
    "Ensemble (2-way)":        ensemble_2way_records,
    "Ensemble (2-way, pooled)": ensemble_2way_pooled_records,
    "Ensemble (3-way)":        ensemble_3way_records,
    "Ensemble (3-way, pooled)": ensemble_3way_pooled_records,
    "L2 model (paired)":       l2_records,
    "L2 model (unpaired)":     l2_baseline_records,
    "Irish+English combined":  l2_combined_records,
    "Irish only (baseline)":   irish_only_records,
}

active_systems = {name: recs for name, recs in SYSTEMS.items() if recs is not None}
print("Active systems:", list(active_systems.keys()))

In [ ]:
rows = []
for system_name, system_records in active_systems.items():
    for r in system_records:
        metrics = evaluate_mispronunciation_detection(
            r['canonical'], r['gold'], r['asr']
        )
        rows.append({
            'system':   system_name,
            'audio_id': r['audio_id'],
            **metrics,
        })

per_utt_df = pd.DataFrame(rows)
per_utt_df

# 5 Aggregate Metrics

Metrics are micro-aggregated: confusion counts (TR, FA, FR, TA) are summed
across all utterances per system, then precision/recall/F1/FAR/FRR are
re-derived from those totals. This gives longer utterances proportionally
more weight, which is what we want since we care about the performance on all phones.

PER is micro-aggregated the same way: total ASR errors / total canonical phones.

In [ ]:
def _derived_metrics(TR, FA, FR, TA, total_errors, total_canonical):
    nan = float('nan')
    precision = TR / (TR + FR)            if (TR + FR) > 0 else nan
    recall    = TR / (TR + FA)            if (TR + FA) > 0 else nan
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else nan)
    far       = FA / (TR + FA)            if (TR + FA) > 0 else nan
    frr       = FR / (FR + TA)            if (FR + TA) > 0 else nan
    per       = total_errors / total_canonical if total_canonical > 0 else nan
    return dict(precision=precision, recall=recall, f1=f1,
                false_acceptance_rate=far, false_rejection_rate=frr, per=per)


agg_rows = []
for system_name, group in per_utt_df.groupby('system', sort=False):
    TR = group['TR'].sum()
    FA = group['FA'].sum()
    FR = group['FR'].sum()
    TA = group['TA'].sum()

    # PER micro-aggregation: recover error counts from per-utterance PER and
    # canonical length (= TR+FA+FR+TA since every canonical position is counted).
    canonical_lengths = group['TR'] + group['FA'] + group['FR'] + group['TA']
    total_canonical   = canonical_lengths.sum()
    total_errors      = (group['per'] * canonical_lengths).sum()

    agg_rows.append({
        'system': system_name,
        'TR': TR, 'FA': FA, 'FR': FR, 'TA': TA,
        'utterances': len(group),
        **_derived_metrics(TR, FA, FR, TA, total_errors, total_canonical),
    })

agg_df = pd.DataFrame(agg_rows).set_index('system')
agg_df

# 6 Results Tables

## 6.1 Summary metrics table

In [ ]:
METRIC_COLS = {
    'precision':             'Precision',
    'recall':                'Recall',
    'f1':                    'F1',
    'false_acceptance_rate': 'FAR',
    'false_rejection_rate':  'FRR',
    'per':                   'PER',
}

summary_table = (
    agg_df[list(METRIC_COLS.keys())]
    .rename(columns=METRIC_COLS)
    .round(3)
)
summary_table

## 6.2 LaTeX export

In [ ]:
print(summary_table.style.to_latex(
    caption="Mispronunciation detection results on FLEURS \\textit{ga\\_ie} evaluation set.",
    label="tab:mdd_results",
    hrules=True,
))

## 6.3 FAR / FRR figure

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

systems = agg_df.index.tolist()
x = np.arange(len(systems))
width = 0.35

bars_far = ax.bar(x - width/2, agg_df['false_acceptance_rate'], width,
                  label='FAR (missed mispronunciations)', color='#d62728')
bars_frr = ax.bar(x + width/2, agg_df['false_rejection_rate'],  width,
                  label='FRR (false alarms)',              color='#1f77b4')

ax.set_ylabel('Rate')
ax.set_ylim(0, 1)
ax.set_xticks(x)
ax.set_xticklabels(systems, rotation=15, ha='right')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
sns.despine(ax=ax)
fig.tight_layout()

fig.savefig(IMAGES_DIR / "far_frr.pdf", bbox_inches='tight')
plt.show()

# 8 Error Analysis

## 8.1 Confusion counts

In [ ]:
counts_table = agg_df[['TR', 'FA', 'FR', 'TA', 'utterances']]
counts_table

## 8.2 Confusion matrix heatmaps

In [ ]:
n = len(active_systems)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5))
if n == 1:
    axes = [axes]

for ax, (system_name, _) in zip(axes, active_systems.items()):
    row = agg_df.loc[system_name]
    matrix = np.array([[row['TR'], row['FA']],
                        [row['FR'], row['TA']]], dtype=int)
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Flagged', 'Accepted'],
                yticklabels=['Mispronounced', 'Correct'])
    ax.set_title(system_name, fontsize=9)
    ax.set_xlabel('Model prediction')
    ax.set_ylabel('Ground truth')

fig.tight_layout()
fig.savefig(IMAGES_DIR / "confusion_matrices.pdf", bbox_inches='tight')
plt.show()

## 8.3 Phone-level data

For each canonical phone position across all utterances, record the ground-truth label and each system's prediction. Used for the per-phoneme breakdown below.

In [ ]:
from scripts.eval.eval import get_canonical_labels

phone_rows = []
for system_name, system_records in active_systems.items():
    for r in system_records:
        gt_labels   = get_canonical_labels(r['canonical'], r['gold'])
        pred_labels = get_canonical_labels(r['canonical'], r['asr'])
        for phone, gt, pred in zip(r['canonical'], gt_labels, pred_labels):
            phone_rows.append({
                'system':        system_name,
                'phone':         phone,
                'mispronounced': gt,    # True = actually wrong (from gold)
                'flagged':       pred,  # True = model flagged it
            })

phone_df = pd.DataFrame(phone_rows)
print(f"{len(phone_df)} phone-position observations across {len(active_systems)} system(s).")

## 8.4 Per-phoneme breakdown

For the top N most frequent canonical phones: how often is each actually mispronounced (gold), and how often does each system flag it?

In [ ]:
TOP_N = 20

# Most common canonical phones across all systems (use first system to avoid duplication)
first_system = next(iter(active_systems))
top_phones = (
    phone_df[phone_df['system'] == first_system]['phone']
    .value_counts()
    .head(TOP_N)
    .index.tolist()
)

# Mispronunciation rate per phone (same across systems — comes from gold)
misp_rate = (
    phone_df[phone_df['system'] == first_system]
    .groupby('phone')['mispronounced']
    .mean()
    .reindex(top_phones)
)

# Detection rate per phone per system
det_rates = {
    name: phone_df[phone_df['system'] == name].groupby('phone')['flagged'].mean().reindex(top_phones)
    for name in active_systems
}

x = np.arange(len(top_phones))
n_systems = len(active_systems)
width = 0.6 / (n_systems + 1)

fig, ax = plt.subplots(figsize=(14, 4))

ax.bar(x, misp_rate, width, label='Mispronounced (gold)', color='#333333', alpha=0.5)
for i, (name, rates) in enumerate(det_rates.items()):
    offset = (i - (n_systems - 1) / 2) * width + width
    ax.bar(x + offset, rates, width, label=f'Flagged — {name}')

ax.set_xticks(x)
ax.set_xticklabels(top_phones, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Rate')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "per_phoneme_breakdown.pdf", bbox_inches='tight')
plt.show()

## 8.5 Ensemble winner distribution

For each span in the ensemble, which model won (ga / en / ru)? Shows how much each model contributes.

In [ ]:
ensemble_variants = {
    name: recs for name, recs in {
        'Ensemble (2-way)': ensemble_2way_records,
        'Ensemble (3-way)': ensemble_3way_records,
    }.items() if recs is not None
}

if not ensemble_variants:
    print("No ensemble records available.")
else:
    winner_data = {}
    for name, recs in ensemble_variants.items():
        counts = {}
        for r in recs:
            for span in r['span_details']:
                w = span['winner']
                counts[w] = counts.get(w, 0) + 1
        winner_data[name] = counts

    models = sorted({m for counts in winner_data.values() for m in counts})
    x = np.arange(len(winner_data))
    width = 0.6 / len(models)
    colours = {'ga': '#2196F3', 'en': '#FF9800', 'ru': '#4CAF50'}

    fig, ax = plt.subplots(figsize=(5, 4))
    for i, model in enumerate(models):
        vals = [winner_data[name].get(model, 0) for name in winner_data]
        offset = (i - (len(models) - 1) / 2) * width
        ax.bar(x + offset, vals, width, label=model, color=colours.get(model))

    ax.set_xticks(x)
    ax.set_xticklabels(list(winner_data.keys()), rotation=10)
    ax.set_ylabel('Spans won')
    ax.legend(title='Model')
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "ensemble_winner_dist.pdf", bbox_inches='tight')
    plt.show()

## 8.6 Confidence score distribution

Histogram of span confidence scores split by whether the ensemble prediction matched canonical. Shows whether confidence is calibrated.

In [ ]:
if ensemble_2way_records is None:
    print("No ensemble records available.")
else:
    correct_conf   = []
    incorrect_conf = []
    for r in ensemble_2way_records:
        for span in r['span_details']:
            if span['canonical'] == span['predicted']:
                correct_conf.append(span['confidence'])
            else:
                incorrect_conf.append(span['confidence'])

    fig, ax = plt.subplots(figsize=(6, 4))
    bins = np.linspace(0, 1, 25)
    ax.hist(correct_conf,   bins=bins, alpha=0.6, label='Correct',   color='#1f77b4', density=True)
    ax.hist(incorrect_conf, bins=bins, alpha=0.6, label='Incorrect', color='#d62728', density=True)
    ax.set_xlabel('Span confidence')
    ax.set_ylabel('Density')
    ax.legend()
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "confidence_dist.pdf", bbox_inches='tight')
    plt.show()
    print(f"Correct spans:   {len(correct_conf):,}  (mean conf {np.mean(correct_conf):.3f})")
    print(f"Incorrect spans: {len(incorrect_conf):,} (mean conf {np.mean(incorrect_conf):.3f})")